In [1]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs  # ml-labs: our experiment management framework

import sys
print(sys.version)

for i in [pd, pl, np, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

from IPython.display import Markdown

from mllabs import Connector, Experimenter, ColSelector
from mllabs.collector import MetricCollector, ModelAttrCollector, StackingCollector
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

# Binary-encode several categorical features using Polars expressions.
# We also derive indicator columns for internet-service-related features
# ('No internet service' is treated as a separate binary flag).
dict_expr = {
    'gender': (pl.col('gender') == 'Male').cast(pl.Int8),
    'No_Internet': (pl.col('DeviceProtection') == 'No internet service').cast(pl.Int8),
    'DSL_Y': (pl.col('InternetService') == 'DSL').cast(pl.Int8)
}
for i in ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService']:
    dict_expr[i] = (pl.col(i) == 'Yes').cast(pl.Int8)

for i in ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'MultipleLines']:
    dict_expr[i + '_Y'] = (pl.col(i) == 'Yes').cast(pl.Int8)

# PolarsLoader reads CSVs into Polars DataFrames.
# ExprProcessor applies the expression dict above in a single vectorized pass.
# PandasConverter converts to pandas, setting 'id' as the index.
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr)
)

df_train = loader.fit_transform([data_path / 'train.csv']).with_columns(
    (pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_test = loader.transform([data_path / 'test.csv'])

# Variable groupings
# X_bin  : original binary features (Yes/No encoded as 0/1)
# X_tri  : 3-level categorical features (Yes / No / No internet service)
# X_bin2 : derived binary flags from X_tri (e.g. OnlineSecurity_Y)
# X_num  : numerical features
# X_nom  : nominal categoricals kept as strings for tree-based models
X_bin = ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService', 'gender', 'SeniorCitizen']
X_tri = ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport']
X_bin2 = ['{}_Y'.format(i) for i in X_tri] + ['No_Internet', 'DSL_Y', 'MultipleLines_Y']
X_tri.append('InternetService')
X_tri.append('MultipleLines')
X_num = ['TotalCharges', 'MonthlyCharges', 'tenure']
X_nom = ['PaymentMethod', 'Contract']
X_all = X_bin + X_tri + X_bin2 + X_num + X_nom

target = 'Churn'

2026-03-14 18:00:22.884673: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-14 18:00:22.930539: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-14 18:00:24.165015: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sun9sun9/python312/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: Fu

3.12.12 (main, Dec 27 2025, 11:08:36) [GCC 13.3.0]
pandas 2.3.3
polars 1.38.1
numpy 2.3.5
matplotlib.pyplot
seaborn 0.13.2
scipy 1.16.3
sklearn 1.8.0
tensorflow 2.20.0
xgboost 3.2.0
lightgbm
catboost 1.2.10
mllabs 0.6.3


In [2]:
if os.path.exists('exp/skf5'):
    e_skf5 = Experimenter.load('exp/skf5', df_train)
    if e_skf5.status == 'closed':
        e_skf5.reopen_exp()
else:
    e_skf5 = Experimenter.create(
        df_train, 'exp/skf5', title='The experimentation using 5-fold StratifiedKFold',
        sp=StratifiedKFold(n_splits=5, shuffle=True, random_state=1),
        splitter_params={'y': target}
    )

e_skf5.set_grp('pre', role='stage', method='transform')
y_edges = {'y': [(None, target)]}
e_skf5.set_grp(
    'clf', role='head', method='predict_proba', edges=y_edges
)
e_skf5.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges=y_edges), slice(-1, None), roc_auc_score, include_train = True
    )
)
e_skf5.add_collector(
    StackingCollector(
        'stacking', Connector(edges=y_edges),
        slice(-1, None), method='mean', experimenter = e_skf5
    )
)
e_skf5.set_grp('xgb', parent='clf', processor=xgb.XGBClassifier,
    adapter=XGBoostAdapter(eval_mode='valid'),
    params={
        'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
        'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 0.01
    }
)

e_skf5.set_grp('lgb', parent='clf', processor=lgb.LGBMClassifier,
    adapter=LightGBMAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)

e_skf5.set_grp('cb', parent='clf', processor=cb.CatBoostClassifier,
    adapter=CatBoostAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': 0, 'random_state': 1, 'max_depth': 5, 'colsample_bylevel': 0.75
    }
)
Markdown(
    e_skf5.desc_spec()
)


Loaded: 9 node(s), 5 group(s), 5 fold(s)


## The experimentation using 5-fold StratifiedKFold

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=5, random_state=1, shuffle=True)` |
| **Inner Splitter (sp_v)** | None |
| **Splitter Params** | `{y='Churn'}` |
| **Outer Folds** | 5 |
| **Inner Folds** | 1 |

In [4]:
params = [(3, 2500, 0.05), (4, 1400, 0.05), (5, 2000, 0.025)]
for i, (max_depth, n_estimators, learning_rate) in enumerate(params):
    e_skf5.set_node(
        'xgb_{}'.format(i), grp = 'xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
        params={'max_depth': max_depth, 'n_estimators': n_estimators, 'learning_rate': learning_rate}
    )
e_skf5.exp()

Experimenting 3 node(s)
Exp 5/5 (100%)                                    
Experimentation complete: 3 node(s)


In [3]:
params = [(7, 2500, 0.05), (15, 2100, 0.025)]
for i, (nl, n_estimators, learning_rate) in enumerate(params):
    e_skf5.set_node(
        'lgb_{}'.format(i), grp = 'lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
        params={'num_leaves': nl, 'n_estimators': n_estimators, 'learning_rate': learning_rate}
    )
e_skf5.exp()

Experimenting 2 node(s)
Exp 5/5 (100%)                                    
Experimentation complete: 2 node(s)


In [4]:
e_skf5.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False)

,valid,train_sub
lgb_0,0.916505,0.918883
lgb_1,0.916439,0.919267


In [7]:
i = 0
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
    params={'colsample_bylevel':0.9, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


In [3]:
i = 1
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
    params={'colsample_bylevel':0.9, 'n_estimators': 1500, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


In [5]:
i = 2
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
    params={'colsample_bylevel':0.9, 'n_estimators': 2500, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


In [7]:
i = 3
e_skf5.set_node(
    'cb_{}'.format(i), grp = 'cb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
    params={'colsample_bylevel':0.9, 'n_estimators': 3500, 'cat_features': ColSelector(col_type = 'category')}
)
e_skf5.exp()

Experimenting 1 node(s)
Exp 5/5 (100%)                 
Experimentation complete: 1 node(s)


In [5]:
e_skf5.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending = False)

,valid,train_sub
xgb_0,0.916613,0.918938
xgb_1,0.916577,0.920044
xgb_2,0.916570,0.921882
cb_3,0.916506,0.919985
lgb_0,0.916505,0.918883
cb_2,0.916440,0.919173
lgb_1,0.916439,0.919267
cb_1,0.916219,0.918076
cb_0,0.915927,0.917255


# Stacking

In [8]:
df_stacking = e_skf5.get_collector('stacking').get_dataset()
df_stacking.head()

xgb_0__Churn_1,cb_3__Churn_1,cb_0__Churn_1,cb_2__Churn_1,xgb_1__Churn_1,lgb_0__Churn_1,xgb_2__Churn_1,lgb_1__Churn_1,cb_1__Churn_1,Churn
f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
0.011076,0.010516,0.010969,0.010733,0.010184,0.009811,0.009709,0.009994,0.010944,0
0.012205,0.012306,0.013141,0.013899,0.011756,0.020308,0.014931,0.017985,0.013012,0
0.959285,0.958362,0.952332,0.956659,0.958874,0.961721,0.957999,0.960321,0.955026,1
0.609279,0.616075,0.61808,0.61755,0.597738,0.629659,0.597232,0.625964,0.61968,0
0.169425,0.163839,0.168902,0.163131,0.167668,0.170073,0.149257,0.167863,0.169181,0


In [12]:
if os.path.exists('exp/stacking'):
    e_stacking = Experimenter.load('exp/stacking', df_stacking)
    if e_stacking.status == 'closed':
        e_stacking.reopen_exp()
else:
    e_stacking = Experimenter.create(
        df_stacking, 'exp/stacking', title='Stacking',
        sp=StratifiedKFold(n_splits=5, random_state=1, shuffle=True),
        splitter_params={'y': target}
    )

e_stacking.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges={'y': [(None, target)]}),
        slice(-1, None),
        roc_auc_score,
        include_train = True
    )
)

e_stacking.set_grp('pre', role='stage', method='transform')
e_stacking.build()
e_stacking.set_grp(
    'clf', role='head', method='predict_proba',
    edges={'y': [(None, target)]}
)
e_stacking.set_grp('lr', parent='clf', processor = LogisticRegression)

📁 Created directory: exp/stacking
Collect 5/5 (100%)        
Building 0 node(s)
Build 5/5 (100%)        
Build complete: 0 node(s)


{'result': 'new',
 'grp': <mllabs._pipeline.PipelineGroup at 0x7793ec6fe4e0>,
 'affected_nodes': []}

In [13]:
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_1', 'xgb_2']]
e_stacking.set_node('lr1', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_2']]
e_stacking.set_node('lr2', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_1', 'xgb_2']]
e_stacking.set_node('lr3', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_2', 'lgb_0']]
e_stacking.set_node('lr4', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_1', 'xgb_2', 'lgb_0', 'lgb_1']]
e_stacking.set_node('lr5', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_1', 'lgb_0']]
e_stacking.set_node('lr6', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_2', 'lgb_0', 'cb_0']]
e_stacking.set_node('lr7', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_2', 'cb_0']]
e_stacking.set_node('lr8', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'lgb_0', 'cb_0']]
e_stacking.set_node('lr9', grp = 'lr', edges = {'X': [(None, X_sel)]})
X_sel = ['{}__Churn_1'.format(i) for i in ['xgb_0', 'xgb_2', 'lgb_0', 'cb_3']]
e_stacking.set_node('lr10', grp = 'lr', edges = {'X': [(None, X_sel)]})
e_stacking.exp()

Experimenting 10 node(s)
Exp 5/5 (100%)                  
Experimentation complete: 10 node(s)


In [14]:
e_stacking.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending=False).iloc[:5]

,valid,train_sub
lr10,0.916783,0.916783
lr4,0.916731,0.916731
lr5,0.916695,0.916695
lr7,0.916688,0.916688
lr6,0.916688,0.916688


# Submission

In [15]:
e_skf5.add_trainer('trainer')
e_skf5.trainers['trainer'].select_head(['xgb_0', 'xgb_2', 'lgb_0', 'cb_3'])
e_skf5.trainers['trainer'].train()

xgb_0 2/2 (100%)                                   
Train complete: 2 node(s)


In [16]:
e_stacking.add_trainer('trainer')
e_stacking.trainers['trainer'].select_head(['lr10'])
e_stacking.trainers['trainer'].train()

lr10 1/1 (100%)                 
Train complete: 1 node(s)


In [21]:
df_result = e_stacking.trainers['trainer'].to_inferencer(slice(-1, None)).process(
    e_skf5.trainers['trainer'].to_inferencer(slice(-1, None)).process(df_test)
)
df_result.columns = ['Churn']

In [22]:
df_result.with_columns(
    df_test['id']
)[['id', 'Churn']].write_csv('data/submission.csv')

In [23]:
# !kaggle competitions submit -c playground-series-s6e3 -f data/submission.csv -m "5"

100%|██████████████████████████████████████| 6.51M/6.51M [00:02<00:00, 2.39MB/s]
Successfully submitted to Predict Customer Churn

In [ ]:
e_skf5.close_exp()